In [78]:
import pandas as pd
import numpy as np
import re
import string

import nltk
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer
from nltk.tag import pos_tag

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer,TfidfVectorizer

from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score,classification_report

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


In [79]:
import pandas as pd

# Read the file as a single column
df = pd.read_csv("Resume.csv", header=None)

# Split each row into columns
df = df[0].str.split(",", n=4, expand=True)

# Set header
df.columns = df.iloc[0]

# Remove header row
df = df.iloc[1:].reset_index(drop=True)

print(df.head())
print(df.columns)

0 resume_id category                                        resume_text  \
0         1       HR  hr administrator marketing associate hr admini...   
1         2       HR  hr specialist us hr operations summary versati...   
2         3       HR  hr director summary over years experience in r...   
3         4       HR  hr specialist summary dedicated driven and dyn...   
4         5       HR  hr manager skill highlights hr skills hr depar...   

0                                        skills_list experience_years  
0  hr administrator marketing associate hr admini...              0.0  
1  hr specialist us hr operations summary versati...              0.0  
2  hr director summary over years experience in r...             20.0  
3  hr specialist summary dedicated driven and dyn...             20.0  
4  hr manager skill highlights hr skills hr depar...              0.0  
Index(['resume_id', 'category', 'resume_text', 'skills_list',
       'experience_years'],
      dtype='object', name=

In [80]:
from sklearn.model_selection import train_test_split

# Features and Target
X = df['resume_text']
y = df['category']

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    # stratify=y
)

print("Training Samples :", len(X_train))
print("Testing Samples  :", len(X_test))

Training Samples : 2284
Testing Samples  : 572


In [81]:
def remove_html(text):
    return re.sub(r'<.*?>','',str(text))

df['resume_text']=df['resume_text'].apply(remove_html)

In [82]:
def remove_punctuation(text):
    return text.translate(str.maketrans('','',string.punctuation))

df['resume_text']=df['resume_text'].apply(remove_punctuation)

In [83]:
contractions={
"can't":"cannot",
"won't":"will not",
"don't":"do not",
"I'm":"I am",
"it's":"it is"
}

def expand(text):
    words=text.split()
    words=[contractions[word] if word in contractions else word for word in words]
    return " ".join(words)

df['resume_text']=df['resume_text'].apply(expand)

In [84]:
stop_words=set(stopwords.words('english'))

def remove_stopwords(text):
    words=word_tokenize(text)
    words=[word for word in words if word.lower() not in stop_words]
    return " ".join(words)

df['resume_text']=df['resume_text'].apply(remove_stopwords)

In [85]:
df['Tokens']=df['resume_text'].apply(word_tokenize)

df[['resume_text','Tokens']].head()

,resume_text,Tokens
0,hr administrator marketing associate hr admini...,"[hr, administrator, marketing, associate, hr, ..."
1,hr specialist us hr operations summary versati...,"[hr, specialist, us, hr, operations, summary, ..."
2,hr director summary years experience recruitin...,"[hr, director, summary, years, experience, rec..."
3,hr specialist summary dedicated driven dynamic...,"[hr, specialist, summary, dedicated, driven, d..."
4,hr manager skill highlights hr skills hr depar...,"[hr, manager, skill, highlights, hr, skills, h..."


In [86]:
stemmer=PorterStemmer()

def stemming(tokens):
    return [stemmer.stem(word) for word in tokens]

df['Stemmed']=df['Tokens'].apply(stemming)

df[['Tokens','Stemmed']].head()

,Tokens,Stemmed
0,"[hr, administrator, marketing, associate, hr, ...","[hr, administr, market, associ, hr, administr,..."
1,"[hr, specialist, us, hr, operations, summary, ...","[hr, specialist, us, hr, oper, summari, versat..."
2,"[hr, director, summary, years, experience, rec...","[hr, director, summari, year, experi, recruit,..."
3,"[hr, specialist, summary, dedicated, driven, d...","[hr, specialist, summari, dedic, driven, dynam..."
4,"[hr, manager, skill, highlights, hr, skills, h...","[hr, manag, skill, highlight, hr, skill, hr, d..."


In [87]:
df['POS']=df['Tokens'].apply(pos_tag)

df[['Tokens','POS']].head()

,Tokens,POS
0,"[hr, administrator, marketing, associate, hr, ...","[(hr, NN), (administrator, NN), (marketing, NN..."
1,"[hr, specialist, us, hr, operations, summary, ...","[(hr, NN), (specialist, NN), (us, PRP), (hr, V..."
2,"[hr, director, summary, years, experience, rec...","[(hr, NNS), (director, NN), (summary, JJ), (ye..."
3,"[hr, specialist, summary, dedicated, driven, d...","[(hr, JJ), (specialist, NN), (summary, NN), (d..."
4,"[hr, manager, skill, highlights, hr, skills, h...","[(hr, NN), (manager, NN), (skill, VB), (highli..."


In [88]:
lemmatizer = WordNetLemmatizer()

def lemmatize(tokens):
    return [lemmatizer.lemmatize(word) for word in tokens]

df['Lemma'] = df['Tokens'].apply(lemmatize)

df[['Tokens','Lemma']].head()

,Tokens,Lemma
0,"[hr, administrator, marketing, associate, hr, ...","[hr, administrator, marketing, associate, hr, ..."
1,"[hr, specialist, us, hr, operations, summary, ...","[hr, specialist, u, hr, operation, summary, ve..."
2,"[hr, director, summary, years, experience, rec...","[hr, director, summary, year, experience, recr..."
3,"[hr, specialist, summary, dedicated, driven, d...","[hr, specialist, summary, dedicated, driven, d..."
4,"[hr, manager, skill, highlights, hr, skills, h...","[hr, manager, skill, highlight, hr, skill, hr,..."


In [89]:
encoder = LabelEncoder()

df['category'] = encoder.fit_transform(df['category'])

df[['category']].head()

,category
0,44
1,44
2,44
3,44
4,44


In [90]:
X = df['resume_text']
y = df['category']

In [91]:
bow = CountVectorizer(max_features=3000)

X_bow = bow.fit_transform(X)

print(X_bow.shape)

(2856, 3000)


In [92]:
tfidf = TfidfVectorizer(max_features=3000)

X_tfidf = tfidf.fit_transform(X)

print(X_tfidf.shape)

(2856, 3000)


In [93]:
import gensim.downloader as api

glove = api.load("glove-wiki-gigaword-100")

[==================================================] 100.0% 128.1/128.1MB downloaded


In [94]:
def glove_vector(text):
    words = text.split()

    vectors = [glove[word] for word in words if word in glove]

    if len(vectors) == 0:
        return np.zeros(100)

    return np.mean(vectors, axis=0)

In [95]:
X_glove = np.array([glove_vector(text) for text in X])

print(X_glove.shape)

(2856, 100)


In [97]:
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf,
    y,
    test_size=0.2,
    random_state=42,
    # stratify=y
)

In [98]:
model = MultinomialNB()

model.fit(X_train, y_train)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


In [99]:
y_pred = model.predict(X_test)

In [100]:
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy :", accuracy)

Accuracy : 0.5244755244755245


In [101]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.00      0.00      0.00         4
           1       0.00      0.00      0.00         1
           3       0.00      0.00      0.00         1
           6       0.00      0.00      0.00         1
           8       0.48      0.94      0.64        17
           9       0.54      0.23      0.32        31
          10       0.00      0.00      0.00        13
          11       0.57      0.27      0.36        15
          12       0.33      0.04      0.07        26
          13       0.00      0.00      0.00         8
          14       0.60      0.68      0.64        22
          15       0.00      0.00      0.00         3
          16       0.00      0.00      0.00         3
          17       0.00      0.00      0.00         4
          18       0.41      0.47      0.44        19
          19       0.00      0.00      0.00         3
          20       0.38      0.73      0.50        22
          21       1.00    

c:\Users\HP\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\HP\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\HP\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mod

In [102]:
resume = """
Python SQL Machine Learning Deep Learning
Data Analysis Pandas NumPy TensorFlow
"""

resume = remove_html(resume)
resume = remove_punctuation(resume)
resume = expand(resume)
resume = remove_stopwords(resume)

resume_vector = tfidf.transform([resume])

prediction = model.predict(resume_vector)

print("Predicted Category :", encoder.inverse_transform(prediction)[0])

Predicted Category : Data Scientist
